In [1]:
from sage.all import *

def metricna_dimenzija_Cn(n, d, prelomi_simetrijo=True):
    """
    Izračuna metrično dimenzijo usmerjenega cirkulantnega grafa C_n(1, ..., n-d).
    Predpostavka: n > 2*d (zato so razdalje le 0, 1 ali 2).
    """
    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen!")
    
    # Dovoljeni koraki naprej (generatorji)
    dovoljeni_koraki = set(range(1, n - d + 1))
    
    # 1. Hitri analitični izračun razdalj d(w, u)
    # d(w, u) je 0 če w == u, 1 če je skok med generatorji, sicer 2
    def razdalja(w, u):
        if w == u:
            return 0
        korak = (u - w) % n
        return 1 if korak in dovoljeni_koraki else 2

    # 2. Nastavitev linearnega programa (ILP)
    lp = MixedIntegerLinearProgram(maximization=False)
    x = lp.new_variable(binary=True)
    
    # Ciljna funkcija: minimiziramo število izbranih radarjev
    lp.set_objective(sum(x[w] for w in range(n)))
    
    # Lomljenje rotacijske simetrije (priporočeno v navodilih za hitrost pri večjih n)
    if prelomi_simetrijo:
        lp.add_constraint(x[0] == 1)

    # 3. Pogoji: vsak par (u, v) mora biti ločen z vsaj enim radarjem
    for u in range(n):
        for v in range(u + 1, n):  # Gledamo samo u < v (brez podvajanja!)
            # Poiščemo vsa vozlišča w, ki ločijo par u in v
            locevalci = [w for w in range(n) if razdalja(w, u) != razdalja(w, v)]
            
            # Dodamo pogoj: vsaj eden izmed ločevalcev mora biti izbran
            lp.add_constraint(sum(x[w] for w in locevalci) >= 1)
            
    # 4. Rešitev modela
    lp.solve()
    
    # Pridobimo izbrana vozlišča (pazimo na zaokroževanje pri float vrednostih)
    razresujoca_mnozica = [w for w in range(n) if round(lp.get_values(x[w])) == 1]
    dimenzija = len(razresujoca_mnozica)
    
    return dimenzija, razresujoca_mnozica